<a href="https://colab.research.google.com/github/aadit6/warwickai-intro-to-ml/blob/main/workshop2/ml_lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workshop 2: Regularisation + F1 Scores

### Getting started
- Save your own copy in Colab.
- Click the **play button** beside a code cell to run it. Work from top to bottom.
- Double-click text to edit an answer. Click outside when done.

The first cell loads four libraries:

| Library | What it does |
|---|---|
| `pandas`, shortened to `pd` | Stores data as a table called a **DataFrame**, like a spreadsheet |
| `numpy`, shortened to `np` | Does maths on a whole list of numbers at once |
| `matplotlib.pyplot`, shortened to `plt` | Draws graphs |
| `sklearn` (scikit-learn) | Ready-made machine learning models and tools |

It then loads a medical dataset that comes with scikit-learn: 442 patients with diabetes, in a table called `diabetes`. The `target` column is how much each patient's disease progressed after a year (higher is worse). The other 10 columns (`age`, `sex`, `bmi`, `bp`, `s1` to `s6`) are inputs, already rescaled to similar sizes.

In [ ]:
import matplotlib.pyplot as plt  # Draws graphs.
import numpy as np  # Does maths on many numbers at once.
import pandas as pd  # Stores data in tables.
from sklearn.datasets import load_diabetes
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

diabetes = load_diabetes(as_frame=True).frame  # Load the data as a table.
diabetes.head()  # Show the first five rows.

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


### 1. Regularisation by hand (3 minutes)

This is the red and blue line example from the slides, a tiny made-up dataset, so the code calls it `toy_x` and `toy_y`. The two training points are $(1, 3)$ and $(2, 5)$.

- **Red line:** slope 2. It hits both points exactly.
- **Blue line:** slope 1. It misses both points slightly, but it is flatter.

Each line gets a **loss** (lower is better) made of two parts:

$$\text{Loss} = \underbrace{\sum (y_{\text{true}} - y_{\text{pred}})^2}_{\text{data fit: how far the line is from the points}} + \underbrace{\lambda \times \text{slope}^2}_{\text{penalty: bigger for steeper lines}}$$

$\lambda$ sets how much we care about the penalty. Replace each `...` to finish `total_loss`.

| Code | What it does |
|---|---|
| `errors ** 2` | Squares every error |
| `np.sum(a)` | Adds up all the values in `a` |
| `slope ** 2` | Squares the slope |
| `lam * a` | Multiplies `a` by lambda |

Expect `0` and `5` at $\lambda = 0$, and `20` and `10` at $\lambda = 5$.

<details><summary>Hint</summary><code>np.sum(errors ** 2)</code> for the data fit, and <code>lam * slope ** 2</code> for the penalty.</details>

In [ ]:
toy_x = np.array([1, 2])
toy_y = np.array([3, 5])

def total_loss(slope, lam):
    predictions = slope * toy_x + 1  # The line's predictions (intercept fixed at 1).
    errors = toy_y - predictions  # One error for each training point.
    data_fit = ...  # Square the errors, then add them up.
    penalty = ...  # Lambda times the slope squared.
    return data_fit + penalty

for lam in [0, 1, 2, 5]:
    print("lambda =", lam, "| red loss:", total_loss(2, lam), "| blue loss:", total_loss(1, lam))

TypeError: unsupported operand type(s) for +: 'ellipsis' and 'ellipsis'

Complete the sentence using the output above: at $\lambda = 0$ the ___ line has the lower loss, at $\lambda = 5$ the ___ line has the lower loss, so a bigger $\lambda$ favours the ___ line.

Your answer: …

Now try every slope from 0 to 3, not just red and blue, and find the best slope for each $\lambda$. There is nothing to fill in: run the cell. The black dot marks the best slope on each curve.

In [ ]:
slopes = np.linspace(0, 3, 301)  # 0.00, 0.01, 0.02, ..., 3.00

for lam in [0, 5, 20, 1000]:
    losses = [total_loss(slope, lam) for slope in slopes]
    best_slope = slopes[np.argmin(losses)]  # np.argmin gives the position of the smallest loss.
    print("lambda =", lam, "| best slope:", round(best_slope, 2))
    if lam < 1000:
        plt.plot(slopes, losses, label=f"lambda = {lam}")
        plt.scatter(best_slope, min(losses), color="black", zorder=3)

plt.xlabel("Slope")
plt.ylabel("Total loss")
plt.ylim(0, 40)
plt.legend()
plt.show()

TypeError: unsupported operand type(s) for +: 'ellipsis' and 'ellipsis'

A line that is almost flat and ignores the trend in the data is **underfitting**: it is too simple.

Complete the sentence using the printed best slopes: as $\lambda$ grows, the best slope gets ___ (bigger or smaller), so the line gets ___ (steeper or flatter).

Your answer: …

### 2. See overfitting

Now a real dataset. The next cell splits the 442 patients into training, validation and test sets (221, 110 and 111 rows). It also turns the 10 inputs into **65** by adding every pair multiplied together, such as `bmi * bp`. More inputs make the model more flexible, so it can overfit more easily. Run it.

In [ ]:
train, rest = train_test_split(diabetes, test_size=0.5, random_state=0)
val, test = train_test_split(rest, test_size=0.5, random_state=0)

input_columns = diabetes.columns.drop("target")  # The names of the 10 input columns.
make_features = PolynomialFeatures(degree=2, include_bias=False)  # Adds the pairs.
scaler = StandardScaler()  # Rescales every input to a similar size.

X_train = scaler.fit_transform(make_features.fit_transform(train[input_columns]))
X_val = scaler.transform(make_features.transform(val[input_columns]))
X_test = scaler.transform(make_features.transform(test[input_columns]))
y_train, y_val, y_test = train["target"], val["target"], test["target"]

print(len(train), len(val), len(test), X_train.shape)  # Rows in each set, then (rows, inputs).

Fit an ordinary `LinearRegression` on all 65 inputs and measure its training and validation MSE. The **baseline** always predicts the average training value. Expect about **1884**, **4176** and **5344**. Lower is better.

| Code | What it does |
|---|---|
| `model.fit(X, y)` | Trains the model on inputs `X` and answers `y` |
| `model.predict(X)` | Uses the trained model to predict answers for `X` |

<details><summary>Hint</summary><code>plain.fit(X_train, y_train)</code>, then copy the training MSE line and change <code>train</code> to <code>val</code> in all three places.</details>

In [ ]:
plain = LinearRegression()
...  # Fit the model here.

plain_train_mse = mean_squared_error(y_train, plain.predict(X_train))
plain_val_mse = ...  # The same, for the validation set.
baseline_mse = mean_squared_error(y_val, np.full(len(y_val), y_train.mean()))

print("training MSE:", round(plain_train_mse), "| validation MSE:", round(plain_val_mse), "| baseline:", round(baseline_mse))

Complete the sentence: the training MSE is ___ (lower or higher) than the validation MSE, so the model is ___ (overfitting or underfitting).

Your answer: …

### 3. Ridge: shrink all the weights (2 minutes)

`Ridge(alpha=...)` is the L2 penalty from the slides, and `alpha` plays the role of $\lambda$. The loop trains one Ridge model for each `alpha`. Complete the validation MSE, then run the cell to plot both lines.

`np.logspace(-2, 4, 25)` makes 25 values from 0.01 up to 10,000, spaced by multiplying, so small and huge values are covered evenly.

**Predict before you run:** what will each line look like as `alpha` grows?

<details><summary>Hint</summary>Copy the training line above it, and change <code>y_train</code> to <code>y_val</code> and <code>X_train</code> to <code>X_val</code>.</details>

In [ ]:
alphas = np.logspace(-2, 4, 25)  # 0.01 ... 10,000
ridge_train = []
ridge_val = []
for alpha in alphas:
    model = Ridge(alpha=alpha)
    model.fit(X_train, y_train)
    ridge_train.append(mean_squared_error(y_train, model.predict(X_train)))
    ridge_val.append(...)  # The same, for the validation set.

plt.plot(alphas, ridge_train, marker="o", label="Training")
plt.plot(alphas, ridge_val, marker="o", label="Validation")
plt.xscale("log")
plt.xlabel("alpha (penalty strength)")
plt.ylabel("MSE")
plt.legend()
plt.show()

Complete the sentences: with a small `alpha` (on the left) the training MSE is ___ (low or high) and the validation MSE is ___ (low or high), which is overfitting. With a very large `alpha` (on the right) both are ___ (low or high), which is underfitting.

Your answer: …

### 4. Lasso: switch weights off

`Lasso(alpha=...)` is the L1 penalty from the slides. Complete the loop to count the weights that are **not** zero, then run the cell to plot both graphs.

| Code | What it does |
|---|---|
| `model.coef_` | The weights the model learned, one per input |
| `weights != 0` | Checks every weight at once: `True` if it is not zero |
| `np.sum(a)` | Counts the `True`s in `a` (each `True` counts as 1) |

<details><summary>Hint</summary><code>np.sum(model.coef_ != 0)</code></details>

In [ ]:
lasso_alphas = np.logspace(-2, 2, 25)  # 0.01 ... 100
lasso_val = []
lasso_nonzero = []
for alpha in lasso_alphas:
    model = Lasso(alpha=alpha, max_iter=20000)  # max_iter gives it enough time to finish.
    model.fit(X_train, y_train)
    lasso_val.append(mean_squared_error(y_val, model.predict(X_val)))
    lasso_nonzero.append(...)  # Count the weights that are not zero.

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
left.plot(lasso_alphas, lasso_val, marker="o")
left.set_xscale("log")
left.set_xlabel("alpha (penalty strength)")
left.set_ylabel("Validation MSE")
right.plot(lasso_alphas, lasso_nonzero, marker="o", color="black")
right.set_xscale("log")
right.set_xlabel("alpha (penalty strength)")
right.set_ylabel("Weights that are not zero")
plt.tight_layout()
plt.show()

Look at the right-hand graph and complete the sentence: as `alpha` grows, the number of non-zero weights gets ___ (bigger or smaller) until it reaches ___, and the model then predicts the ___ for every patient.

Your answer: …

### 5. F1 Scores

We are using measurements from breast cancer samples to predict whether a tumour is malignant or benign. The dataset used contains measurements taken from breast cancer samples.

We treat malignant as the positive class.

A model can make two types of mistakes:
- False negative: A malignant tumour is predicted to be benign.
- False positive: A benign tumour is predicted to be malignant.

We treat malignant as the positive class.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# Load the dataset.
data = load_breast_cancer(as_frame=True)


X = data.data #This is the data about the cancer, i.e. different features
y = data.target #These are the TRUE labels of the data, 0 = malignant, 1 = benign. It's an array of 0s and 1s

# Show the first five rows.
X.head()

Each row represents one sample, and each column in X represents a feature, such as the radius, texture, perimeter or area of a tumour.

**Questions:**



1.   What is the difference between X and y?
2.   When we create our model, how is it going to classify the data, i.e. how will it label malignant vs benign tumours? What will y_pred (the predicted data) look like?

**What to add to the code below:**

1.   Fill in the gap so that the test data is 25% of the total data.


```
model.fit(???, ???)
```


2. What goes in the gap here? (What do we need to train the data?)
3. What goes in the gap here? (What are we using to predict the labels of the data?)

```
y_pred = model.predict(???)
```




In [ ]:
# Split the data into training and testing sets.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=...,
    random_state=42,
    stratify=y #Preserves the proportions of each target class in the training and test sets
)

C = 0.001 #C is the inverse of regularisation strength.

# Train a logistic regression classifier.
model = LogisticRegression(C=C,max_iter=5000)
model.fit(..., ...)

# Predict the test labels.
y_pred = model.predict(...) #These are the PREDICTED labels, also just 0s and 1s

print("Number of training samples:", len(X_train))
print("Number of test samples:", len(X_test))

**Question**
1. Can you explain what X_train, X_test, y_train and y_test are and what the difference between them are?
2. What goes in the gaps here?


```
f1 = ???(???, ???, pos_label=0)
```




3.   Now, try changing the value of C and see how the F1 score changes. Try really extreme values to see that when one of precision/recall is 0, the F1 score becomes 0 too.




In [ ]:
# Calculate evaluation metrics.
accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test, y_pred, pos_label=0
) #Here the positive value has label 0, so we need to set pos_label = 0

recall = recall_score(y_test, y_pred, pos_label=0)

f1 = ...(..., ..., pos_label=0)

print(f"Accuracy:  {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1 score:  {f1:.3f}")